# TechStore Recommender Training

Notebook nay tao synthetic behavior co pattern ro rang, sau do train/load lai cac model sequence de metric tang len de quan sat de hon.

## Cach dung

Chay Jupyter trong thu muc `ai-service`:

```bash
cd /home/hoang/django_projects/final/TechStore/ai-service
source .venv/bin/activate
jupyter notebook
```

Mo file nay, dam bao kernel la `Python (.venv) TechStore AI`, roi bam `Run All`.

In [22]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "app").exists():
    raise RuntimeError("Hay mo Jupyter voi cwd la thu muc ai-service.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working directory: {PROJECT_ROOT}")

Working directory: /home/hoang/django_projects/final/TechStore/ai-service


In [ ]:
from app.synthetic_behavior import generate_synthetic_behavior

synthetic_path = PROJECT_ROOT / "data" / "user_behavior_synthetic.csv"
generate_synthetic_behavior(synthetic_path, num_users=120, sequence_repeats=18, seed=42)

os.environ["BEHAVIOR_FILE"] = str(synthetic_path)
os.environ["SEQUENCE_EPOCHS"] = "12"

print(f"Using behavior file: {synthetic_path}")
pd.read_csv(synthetic_path).head()

In [ ]:
from app.recommendation import Recommender

recommender = Recommender()
recommender.model_metrics

In [ ]:
rows = []
for model_name, metrics in recommender.model_metrics.items():
    rows.append(
        {
            "model": model_name.upper(),
            "top1_accuracy": float(metrics["top1_accuracy"]),
            "top5_accuracy": float(metrics["top5_accuracy"]),
        }
    )

metrics_df = pd.DataFrame(rows).sort_values(
    ["top1_accuracy", "top5_accuracy"],
    ascending=False,
).reset_index(drop=True)

for column in ["top1_accuracy", "top5_accuracy"]:
    metrics_df[column] = metrics_df[column].astype(float).round(6)

display(metrics_df)

In [ ]:
best_name = metrics_df.loc[0, "model"]
print(f"model_best = {best_name}")

In [ ]:
result = recommender.recommend(user_id=1, limit=5)
pd.DataFrame(result["items"])[["id", "name", "price", "score"]]